In [2]:
import os
# Load drive folder
folder_path = f'data'

files_path = os.listdir(folder_path)
print("Files in folder:", files_path)

Files in folder: ['lamborghini', 'bugatti', 'ferrari', 'maserati']


In [3]:
import pandas as pd

df = pd.read_excel("Top 5 Links x Model.xlsx")
df.head()

,Hashtag,item_id,TikTok Link,Start Week Day,End Week Day,Video views,Engagement rate,Like count,Comment count,Share count
0,abarth595,7415284006092033312,https://www.tiktok.com/@andrea_petrini/video/7...,2024-09-16 00:00:00,2025-04-20,3690680,0.027615,50119,876,50923
1,abarth595,7417509751627549985,https://www.tiktok.com/@andrea_petrini/video/7...,2024-09-16,2025-01-26,2132603,0.035900,59827,1442,15292
2,abarth595,7422333790560849185,https://www.tiktok.com/@andrea_petrini/video/7...,2024-09-30,2025-04-27,2114425,0.019666,34853,1265,5464
3,abarth595,7492491904890752278,https://www.tiktok.com/@andrea_petrini/video/7...,2025-04-07,2025-04-27,2056133,0.033904,62191,1113,6408
4,abarth595,7457983608406543638,https://www.tiktok.com/@andrea_petrini/video/7...,2025-01-06,2025-04-27,1978585,0.019804,36636,913,1634


In [4]:
import re
# List of known brands (lowercased for consistency)
BRANDS = ["ferrari", "lamborghini", "maserati", "bugatti"]

def extract_brand(model: str) -> str:
    model = re.sub(r'[^a-zA-Z0-9]', '', model.lower())
    for brand in BRANDS:
        if brand in model:
            return brand
    return "Unknown"

# Apply to DataFrame
df["brand"] = df["Hashtag"].apply(extract_brand)

# Adding data for files
df['audio_path'] = 'data/' + df["brand"] + "/" + df['item_id'].astype(str) + '.mp3'
filtered_df = df[df["brand"]!="Unknown"]
filtered_df

,Hashtag,item_id,TikTok Link,Start Week Day,End Week Day,Video views,Engagement rate,Like count,Comment count,Share count,brand,audio_path
130,bugattichiron,7443430790769200392,https://www.tiktok.com/@kcars_/video/744343079...,2024-11-25 00:00:00,2025-05-25 00:00:00,8547294,0.216501,1701922,11249,137323,bugatti,data/bugatti/7443430790769200392.mp3
131,bugattichiron,7440895350363180309,https://www.tiktok.com/@jacobandco/video/74408...,2024-11-18 00:00:00,2025-05-25 00:00:00,8080215,0.050086,395732,2197,6775,bugatti,data/bugatti/7440895350363180309.mp3
132,bugattichiron,7424429150057844000,https://www.tiktok.com/@omid_hypercars/video/7...,2024-10-07 00:00:00,2025-04-13 00:00:00,6800099,0.037574,254029,514,967,bugatti,data/bugatti/7424429150057844000.mp3
133,bugattichiron,7410825202000547093,https://www.tiktok.com/@drivttx/video/74108252...,2024-09-02 00:00:00,2025-05-25 00:00:00,5038014,0.029950,141746,1148,7996,bugatti,data/bugatti/7410825202000547093.mp3
134,bugattichiron,7490948226145668374,https://www.tiktok.com/@carspott_kga.msk/video...,2025-04-07 00:00:00,2025-05-25 00:00:00,5021139,0.097847,415309,5331,70661,bugatti,data/bugatti/7490948226145668374.mp3
...,...,...,...,...,...,...,...,...,...,...,...,...
753,maseratimc20,7429261279149264161,https://www.tiktok.com/@rseitalia/video/742926...,2024-10-21 00:00:00,2025-05-25 00:00:00,9014806,0.002410,21271,43,411,maserati,data/maserati/7429261279149264161.mp3
754,maseratimc20,7479727028690881814,https://www.tiktok.com/@rseitalia/video/747972...,2025-03-31 00:00:00,2025-05-25 00:00:00,6568321,0.001381,8632,102,340,maserati,data/maserati/7479727028690881814.mp3
755,maseratimc20,7486387979439656214,https://www.tiktok.com/@rseitalia/video/748638...,2025-03-31 00:00:00,2025-05-25 00:00:00,4980194,0.000951,4552,40,145,maserati,data/maserati/7486387979439656214.mp3
756,maseratimc20,7502466608384707862,https://www.tiktok.com/@thesatinblackcollectio...,2025-05-05 00:00:00,2025-05-25 00:00:00,2954838,0.158045,443527,960,22510,maserati,data/maserati/7502466608384707862.mp3


In [10]:
import whisper

model = whisper.load_model("small")

filtered_df['transcription'] = ''

for index, row in filtered_df.iterrows():

    try:
        result = model.transcribe(row['audio_path'], fp16=False)
        filtered_df.at[index, 'transcription'] = result['text']
    except Exception as e:
        print(f"Something went wrong: {e}")

/tmp/ipykernel_86457/212258497.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['transcription'] = ''


In [11]:
filtered_df.to_csv("data_filtered.csv", index=False)